# 🔷 CryptoHedgeLab — Master Backtest Engine v4

**9 Algorithmic Strategies | BTC/ETH | Hourly Data | 2020–Present**

---

This notebook implements and backtests nine distinct crypto trading strategies ranging from simple momentum to delta-neutral derivatives hedging. Each strategy includes:
- Clear entry and exit signal logic
- Dynamic position sizing via ATR
- Transaction cost modelling (3 bps per trade)
- Sharpe ratio, CAGR, max drawdown, and win rate metrics
- Publication-quality equity curve charts

### Strategies Overview

| # | Strategy | Asset | Type | Edge |
|---|----------|-------|------|------|
| 1 | **Momentum** | BTC | Trend | EMA crossover + ADX + RSI filter |
| 2 | **Funding Arbitrage** | BTC | Carry | Basis premium carry in bull regime |
| 3 | **Pairs Trading** | BTC/ETH | Stat-Arb | Rolling correlation spread mean-reversion |
| 4 | **Dual Momentum** | BTC | Trend | SMA cross + MACD + ADX-scaled sizing |
| 5 | **Margin Short** | BTC | Counter-trend | Bear-regime RSI exhaustion shorts |
| 6 | **Vol Straddle** | ETH | Volatility | BB squeeze breakout on vol expansion |
| 7 | **Perp Swap Hedge** | BTC | Carry | Delta-neutral funding rate harvesting |
| 8 | **Inverse Perp Hedge** | BTC | Hedge | Spot long + inverse perp short overlay |
| 9 | **Trend+Vol Filter** | BTC | Trend | EMA stack + vol-spike exit for live safety |

## ⚙️ Setup & Dependencies

Install required libraries if not already present:
```bash
pip install ccxt pandas numpy matplotlib
```

In [ ]:
import ccxt
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import warnings, os

warnings.filterwarnings('ignore')
os.makedirs('ui/public/backtest_results', exist_ok=True)
print('✅ Imports OK')

## 📡 Data Fetching

We fetch hourly OHLCV data from Binance via the `ccxt` library.
- **BTC/USDT** — primary asset for most strategies
- **ETH/USDT** — used for pairs trading (S3) and vol straddle (S6)
- Start date: `2023-01-01` (adjust `since` to e.g. `2020-01-01` for longer history)

> ⚠️ Binance International may be geo-restricted. The fetcher automatically falls back to Binance US.

In [ ]:
def get_exchange():
    try:
        ex = ccxt.binance({'enableRateLimit': True})
        ex.load_markets()
        return ex
    except Exception:
        print('Binance restricted → Binance US')
        return ccxt.binanceus({'enableRateLimit': True})

exchange = get_exchange()

def fetch_data(symbol, timeframe='1h', since='2023-01-01T00:00:00Z'):
    print(f'  → {symbol} [{timeframe}]...')
    try:
        since_ms = exchange.parse8601(since)
        all_ohlcv = []
        while since_ms < exchange.milliseconds():
            try:
                ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since_ms, limit=1000)
            except Exception:
                if 'USDT' in symbol:
                    symbol = symbol.replace('USDT', 'USD')
                    ohlcv = exchange.fetch_ohlcv(symbol, timeframe, since_ms, limit=1000)
                else:
                    raise
            if not ohlcv: break
            since_ms = ohlcv[-1][0] + 1
            all_ohlcv += ohlcv
        df = pd.DataFrame(all_ohlcv, columns=['timestamp','open','high','low','close','volume'])
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
        df.set_index('timestamp', inplace=True)
        df = df[~df.index.duplicated(keep='first')]
        print(f'    ✓ {len(df):,} bars')
        return df
    except Exception as e:
        print(f'  ✗ {e}'); return pd.DataFrame()

print('📡 Fetching data...')
btc = fetch_data('BTC/USDT', '1h', '2023-01-01T00:00:00Z')
eth = fetch_data('ETH/USDT', '1h', '2023-01-01T00:00:00Z')
print(f'\nBTC: {len(btc):,} bars | ETH: {len(eth):,} bars')

## 📐 Technical Indicators

All indicators are implemented from scratch using pandas/numpy for full transparency.

| Indicator | Formula Summary |
|-----------|----------------|
| **RSI** | Wilder's smoothed EMA of gains/losses, 14-period |
| **ATR** | Exponential average of True Range (max of H-L, H-Cprev, L-Cprev) |
| **ADX** | Directional Movement Index — measures trend strength |
| **MACD** | EMA(12) − EMA(26), signal = EMA(9) of MACD |
| **Bollinger Bands** | 20-period SMA ± 2σ |
| **Z-Score** | Rolling (x − μ) / σ over a window |

In [ ]:
def compute_rsi(series, period=14):
    delta = series.diff()
    g = delta.clip(lower=0)
    l = -delta.clip(upper=0)
    ag = g.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    al = l.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    return 100 - (100 / (1 + ag / al.replace(0, np.nan)))

def compute_atr(df, period=14):
    hl = df.high - df.low
    hc = (df.high - df.close.shift()).abs()
    lc = (df.low  - df.close.shift()).abs()
    return pd.concat([hl, hc, lc], axis=1).max(axis=1).ewm(span=period, adjust=False).mean()

def compute_adx(df, period=14):
    h, l, c = df.high, df.low, df.close
    pdm = h.diff().clip(lower=0).where(h.diff() > -l.diff(), 0)
    mdm = (-l.diff()).clip(lower=0).where(-l.diff() > h.diff(), 0)
    atr = compute_atr(df, period)
    pdi = 100 * pdm.ewm(span=period, adjust=False).mean() / atr
    mdi = 100 * mdm.ewm(span=period, adjust=False).mean() / atr
    dx  = (100 * (pdi - mdi).abs() / (pdi + mdi).replace(0, np.nan))
    return dx.ewm(span=period, adjust=False).mean().fillna(20), pdi, mdi

def compute_macd(s, fast=12, slow=26, sig=9):
    ef = s.ewm(span=fast, adjust=False).mean()
    es = s.ewm(span=slow, adjust=False).mean()
    ml = ef - es
    sl = ml.ewm(span=sig, adjust=False).mean()
    return ml, sl, (ml - sl).fillna(0)

def compute_bbands(s, period=20, std=2):
    mu = s.rolling(period).mean()
    sigma = s.rolling(period).std()
    return mu + std*sigma, mu, mu - std*sigma

def compute_zscore(s, window=50):
    mu  = s.rolling(window).mean()
    sig = s.rolling(window).std()
    return (s - mu) / sig.replace(0, np.nan)

def volume_ma(df, period=20):
    return df.volume.rolling(period).mean()

print('✅ Indicators defined')

## 🔧 Backtest Engine

### Position Builder
Converts raw boolean signal arrays into a position array `{-1, 0, +1}` with proper state machine logic — no look-ahead bias, no simultaneous long+short.

### Equity Builder
Computes bar-by-bar P&L using **lagged positions** (position from previous bar drives this bar's return). Deducts 3 bps transaction cost on every position change.

### Dynamic Sizing
Position size scales inversely with ATR: larger size in low-volatility regimes, smaller in high-vol. Capped at 3× base size. This implements a form of volatility targeting.

```
size = (price × risk_pct) / (atr_mult × ATR)
size_normalised = size / mean(size)
```

In [ ]:
def dynamic_size(df, risk_pct=0.01, atr_period=14, atr_mult=2.0):
    if 'high' in df.columns and 'low' in df.columns:
        atr = compute_atr(df, atr_period)
    else:
        atr = df.close.pct_change().rolling(atr_period).std() * df.close
        atr = atr.bfill()
    size = (df.close * risk_pct) / (atr_mult * atr)
    size = size.clip(upper=3)
    mean = size.mean()
    return (size / mean if mean > 0 else size).fillna(1.0)

def build_positions(buy_arr, sell_arr, allow_short=False):
    """State-machine position builder. No ffill, no look-ahead."""
    pos = np.zeros(len(buy_arr))
    cur = 0
    for i in range(len(buy_arr)):
        if   cur == 0  and buy_arr[i]:  cur = 1
        elif cur == 1  and sell_arr[i]: cur = -1 if allow_short else 0
        elif cur == -1 and buy_arr[i]:  cur = 0
        pos[i] = cur
    return pos

def build_equity(df, positions, size_series=None, price_col='close', alt_col=None):
    """Bar-by-bar equity with lagged positions and transaction costs."""
    pr  = df[price_col].pct_change().fillna(0).values
    pl  = np.roll(positions, 1); pl[0] = 0
    sr  = pl * (pr - df[alt_col].pct_change().fillna(0).values) if alt_col else pl * pr
    if size_series is not None:
        sz = np.roll(size_series.values, 1); sz[0] = 1
        sr = sr * sz
    sr -= np.abs(np.diff(positions, prepend=0)) * 0.0003  # 3 bps per trade
    return pd.Series(np.cumprod(1 + sr), index=df.index)

def compute_metrics(equity, positions, freq=8760):
    r      = equity.pct_change().fillna(0)
    sharpe = (r.mean() / r.std() * np.sqrt(freq)) if r.std() > 0 else 0
    dd     = ((equity - equity.cummax()) / equity.cummax()).min()
    roi    = (equity.iloc[-1] - 1) * 100
    years  = len(equity) / freq
    cagr   = ((equity.iloc[-1] ** (1/years)) - 1) * 100 if years > 0 else 0
    p = positions; in_t = False; e_eq = 1.0; wins = tot = 0
    for i in range(len(p)):
        if not in_t and p[i] != 0:
            in_t = True; e_eq = equity.iloc[i]
        elif in_t and p[i] == 0:
            in_t = False; tot += 1; wins += (equity.iloc[i] > e_eq)
    return {
        'ROI (%)':      round(roi, 2),
        'CAGR (%)':     round(cagr, 2),
        'Sharpe':       round(sharpe, 2),
        'Max DD (%)':   round(dd * 100, 2),
        'Win Rate (%)': round(wins/tot*100 if tot else 0, 2),
        'Num Trades':   tot
    }

print('✅ Backtest engine defined')

## 🎨 Chart Style & Plot Function

All charts use a pure black background with a dark panel for axes. Entry/exit markers are derived exclusively from **position array transitions** (not raw signal arrays) — this ensures the marker count matches the actual trade count exactly.

Each chart has two panels:
1. **Top**: Price with buy (▲) and sell (▼) markers, plus metrics overlay
2. **Bottom**: Cumulative return % (green) with drawdown % shaded in red

In [ ]:
S = {
    'bg': '#000000', 'panel': '#0a0a0a', 'border': '#1a1a2e',
    'text': '#8892a4', 'bright': '#e2e8f0',
    'blue': '#38bdf8', 'purple': '#a78bfa',
    'green': '#34d399', 'red': '#f87171', 'yellow': '#fbbf24',
}

def apply_style():
    plt.rcParams.update({
        'axes.facecolor': S['panel'], 'figure.facecolor': S['bg'],
        'text.color': S['text'], 'axes.labelcolor': S['text'],
        'xtick.color': S['border'], 'ytick.color': S['border'],
        'axes.edgecolor': S['border'], 'grid.color': S['border'],
        'grid.alpha': 0.35, 'font.family': 'monospace',
        'axes.grid': True, 'grid.linestyle': '--',
    })

def plot_strategy(df, positions, equity, metrics, name, asset_names):
    """
    positions: 1-D numpy array of {-1, 0, 1}
    Markers derived from POSITION TRANSITIONS only.
    """
    apply_style()
    pos_diff = np.diff(positions, prepend=0)
    buy_idx  = np.where(pos_diff > 0)[0]
    sell_idx = np.where(pos_diff < 0)[0]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(16, 9),
        gridspec_kw={'height_ratios': [3, 2]},
        facecolor=S['bg']
    )
    fig.subplots_adjust(hspace=0.06)
    for ax in (ax1, ax2):
        ax.set_facecolor(S['panel'])
        for sp in ax.spines.values(): sp.set_edgecolor(S['border'])

    ax1.plot(df.index, df.close, color=S['blue'], lw=1.4, label=asset_names[0], alpha=0.9)
    if len(asset_names) > 1 and 'close_alt' in df.columns:
        ax1r = ax1.twinx()
        ax1r.plot(df.index, df.close_alt, color=S['purple'], lw=1, alpha=0.55, label=asset_names[1])
        ax1r.set_ylabel(asset_names[1], color=S['purple'], fontsize=9)
        ax1r.tick_params(colors=S['border'])

    if len(buy_idx):
        bdf = df.iloc[buy_idx]
        ax1.scatter(bdf.index, bdf.close, marker='^', color=S['green'], s=70,
                    zorder=5, edgecolors='white', lw=0.4, label='BUY/COVER')
    if len(sell_idx):
        sdf2 = df.iloc[sell_idx]
        ax1.scatter(sdf2.index, sdf2.close, marker='v', color=S['red'], s=70,
                    zorder=5, edgecolors='white', lw=0.4, label='SELL/SHORT')

    m   = metrics
    txt = (f'  ROI: {m["ROI (%)"]:.1f}%   CAGR: {m["CAGR (%)"]:.1f}%\n'
           f'  Sharpe: {m["Sharpe"]:.2f}   Max DD: {m["Max DD (%)"]:.1f}%\n'
           f'  Win Rate: {m["Win Rate (%)"]:.1f}%   Trades: {m["Num Trades"]}')
    ax1.text(0.01, 0.97, txt, transform=ax1.transAxes, va='top', fontsize=9.5,
             color=S['bright'], family='monospace',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#000000', alpha=0.85, edgecolor=S['border']))
    ax1.set_title(f'◆  {name.upper().replace("_"," ")}  |  {"/".join(asset_names)}',
                  color=S['bright'], fontsize=14, pad=12, weight='bold')
    ax1.set_ylabel('Price (USDT)', color=S['text'])
    ax1.legend(loc='upper right', facecolor=S['bg'], edgecolor=S['border'],
               fontsize=8, labelcolor=S['bright'])
    ax1.set_xticklabels([])
    ax1.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'${v:,.0f}'))

    pct    = (equity - 1) * 100
    dd_pct = ((equity - equity.cummax()) / equity.cummax()) * 100
    ax2.fill_between(equity.index, 0, pct, where=pct >= 0, color=S['green'], alpha=0.18)
    ax2.fill_between(equity.index, 0, pct, where=pct <  0, color=S['red'],   alpha=0.18)
    ax2.plot(equity.index, pct, color=S['green'], lw=2, label='Cumulative Return %')
    ax2.fill_between(equity.index, dd_pct, 0, color=S['red'], alpha=0.25, label='Drawdown %')
    ax2.plot(equity.index, dd_pct, color=S['red'], lw=0.8, alpha=0.6)
    ax2.axhline(0, color=S['border'], lw=1)
    ax2.set_ylabel('Return / Drawdown %', color=S['text'], fontsize=9)
    ax2.set_xlabel('Date', color=S['text'])
    ax2.legend(loc='upper left', facecolor=S['bg'], edgecolor=S['border'],
               fontsize=8, labelcolor=S['bright'])
    ax2.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.0f}%'))
    ax2.tick_params(axis='x', colors=S['text'], labelsize=7, rotation=15)

    plt.savefig(f'ui/public/backtest_results/{name}.png',
                format='png', dpi=150, bbox_inches='tight', facecolor=S['bg'])
    plt.close(fig)
    print(f'  ✓ {name}.png  |  ROI: {m["ROI (%)"]:.1f}%  Sharpe: {m["Sharpe"]:.2f}  Trades: {m["Num Trades"]}')
    return metrics

all_metrics = {}
print('✅ Plotting engine defined')

---
## Strategy 1 — Momentum

### Concept
A classic trend-following strategy that enters when multiple momentum indicators align in a bull regime, and exits on the first sign of trend deterioration.

### Buy Signal (ALL conditions must be true)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Rate of Change | ROC(12) > 1.5% | Positive short-term momentum |
| EMA stack | Close > EMA20 > EMA50 | Bullish trend structure |
| ADX | ADX > 22 | Strong directional movement |
| Volume | Volume > 1.1 × VolMA(20) | Momentum confirmed by volume |
| RSI | RSI < 75 | Not yet overbought |

### Sell Signal (ANY condition triggers exit)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Price breakdown | Close < EMA20 | Short-term trend broken |
| ADX decay | ADX < 17 | Trend losing strength |
| Overbought | RSI > 82 | Exhaustion signal |

### Sizing
ATR-based dynamic sizing with 1.2% risk per trade, 2× ATR stop-loss distance.

In [ ]:
print('[1/9] MOMENTUM')
df = btc.copy()
df['ema20'] = df.close.ewm(span=20).mean()
df['ema50'] = df.close.ewm(span=50).mean()
df['roc']   = df.close.pct_change(12)
df['volma'] = volume_ma(df, 20)
adx, _, _   = compute_adx(df, 14)
df['adx']   = adx.fillna(20)
df['rsi']   = compute_rsi(df.close, 14)

# ── BUY: momentum + trend + volume aligned ──────────────────────────
bc = ((df.roc > 0.015)
      & (df.close > df.ema20) & (df.ema20 > df.ema50)
      & (df.adx > 22)
      & (df.volume > df.volma * 1.1)
      & (df.rsi < 75))

# ── SELL: trend or momentum deteriorates ────────────────────────────
sc = (df.close < df.ema20) | (df.adx < 17) | (df.rsi > 82)

pos = build_positions(bc.astype(int).values, sc.astype(int).values)
eq  = build_equity(df, pos, size_series=dynamic_size(df, 0.012, 14, 2.0))
all_metrics['Momentum'] = compute_metrics(eq, pos)
plot_strategy(df, pos, eq, all_metrics['Momentum'], 'momentum', ['BTC/USDT'])

---
## Strategy 2 — Funding Arbitrage (Basis Carry)

### Concept
In crypto markets, perpetual futures consistently trade at a **premium to spot** during bull markets because leveraged longs dominate. The funding rate (paid every 8 hours, positive when premium > 0) acts as a yield stream for delta-neutral shorts.

We model this via the **price basis**: `basis = close − EMA(200)`. When basis is positively elevated (z-score > 0.8), spot is trading above fair value → perpetual funding is likely positive → holding spot + short perp earns carry.

Since we only trade spot here, the strategy is:
- Enter long spot when basis is elevated AND we're in a bull regime
- Exit when the premium collapses (basis reverts to EMA) or regime shifts

> ⚠️ **Why the old version failed**: The original strategy bought on *negative* basis z-score (i.e., dip-buying). This is mean-reversion, not funding arb. In bear markets, oversold conditions persist for months, creating massive drawdowns.

### Buy Signal (ALL conditions must be true)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Basis Z-Score | zb > 0.8 | Spot at premium → positive funding |
| Regime | SMA50 > SMA200 | Confirmed bull market |
| RSI | 45 < RSI < 75 | Sustainable trend, not extreme |
| Basis expanding | basis > basis(24h ago) | Premium still growing |

### Sell Signal (ANY condition triggers exit)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Basis collapses | zb < 0.0 | Funding premium gone |
| Regime shift | SMA50 < SMA200 | Bear market — exit long |
| Parabolic | RSI > 80 | Funding often reverses near tops |

### Sizing
Conservative 0.8% risk (carry strategies have thin but consistent margins).

In [ ]:
print('[2/9] FUNDING ARBITRAGE (BASIS CARRY)')
df = btc.copy()
df['ema200'] = df.close.ewm(span=200).mean()
df['sma50']  = df.close.rolling(50).mean()
df['sma200'] = df.close.rolling(200).mean()
df['basis']  = df.close - df.ema200
df['zb']     = compute_zscore(df.basis, 200)   # wider window = more stable
df['rsi']    = compute_rsi(df.close, 14)
df['basis_exp'] = df.basis > df.basis.shift(24) # expanding premium over 24h

# ── BUY: positive basis (spot premium) + bull regime ────────────────
bc = ((df.zb > 0.8)
      & (df.sma50 > df.sma200)           # bull regime confirmed
      & df.rsi.between(45, 75)           # sustainable, not extreme
      & df.basis_exp)                    # premium still expanding

# ── SELL: premium collapsed OR regime shift OR parabolic ────────────
sc = ((df.zb < 0.0)
      | (df.sma50 < df.sma200)
      | (df.rsi > 80))

pos = build_positions(bc.astype(int).values, sc.astype(int).values)
eq  = build_equity(df, pos, size_series=dynamic_size(df, 0.008, 14, 2.0))
all_metrics['Funding Arb'] = compute_metrics(eq, pos)
plot_strategy(df, pos, eq, all_metrics['Funding Arb'], 'funding_arbitrage', ['BTC/USDT'])

---
## Strategy 3 — Pairs Trading (BTC/ETH Statistical Arbitrage)

### Concept
BTC and ETH are highly correlated assets. When their price ratio deviates significantly from its rolling mean, it tends to revert. We model this via a **cointegration spread** using a rolling correlation hedge ratio.

```
spread = log(BTC) − β × log(ETH)
β = rolling_correlation × (σ_BTC / σ_ETH)   [200-bar window]
zscore = (spread − mean) / std
```

### Buy Signal — Long Spread (BTC cheap relative to ETH)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Z-Score | z < −2.8 | BTC deeply undervalued vs ETH |

### Sell Signal — Close Long
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Mean reversion | z(prev3) < −2.8 AND z > −2.8 | Spread crossed back (confirmed reversion) |
| Stop-loss | z < −4.5 | Regime change — spread not reverting |

### Short Signal — Short Spread (BTC expensive relative to ETH)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Z-Score | z > +2.8 | BTC deeply overvalued vs ETH |

### Cover Signal
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Mean reversion | z(prev3) > +2.8 AND z < +2.8 | Spread crossed back |
| Stop-loss | z > +4.5 | Regime change |

> Long and short positions are clipped to `[-1, 1]` — never double-exposed.

In [ ]:
print('[3/9] PAIRS TRADING')
df_pair = pd.concat([btc.close, eth.close], axis=1).dropna()
df_pair.columns = ['close', 'close_alt']

W = 200
log_b = np.log(df_pair.close)
log_e = np.log(df_pair.close_alt)

# Rolling correlation hedge ratio (more stable than OLS on non-stationary series)
corr  = log_b.rolling(W).corr(log_e)
std_b = log_b.rolling(W).std()
std_e = log_e.rolling(W).std()
beta  = (corr * std_b / std_e.replace(0, np.nan)).fillna(1.0).clip(0.3, 3.0)

spread       = log_b - beta * log_e
df_pair['z'] = compute_zscore(spread, W)
z_prev3      = df_pair.z.shift(3)

# Mean-reversion confirmation: z crossed back through ±2.8
recovering_long  = (z_prev3 < -2.8) & (df_pair.z > -2.8)
recovering_short = (z_prev3 >  2.8) & (df_pair.z <  2.8)

# ── LONG SPREAD: BTC cheap vs ETH ───────────────────────────────────
le = (df_pair.z < -2.8)
lx = recovering_long | (df_pair.z < -4.5)    # reversion OR hard stop

# ── SHORT SPREAD: BTC expensive vs ETH ─────────────────────────────
se = (df_pair.z >  2.8)
sx = recovering_short | (df_pair.z >  4.5)   # reversion OR hard stop

lp = build_positions(le.astype(int).values, lx.astype(int).values)
sp = -build_positions(se.astype(int).values, sx.astype(int).values)
pos_pairs = np.clip(lp + sp, -1, 1)          # ✅ never double-exposed

eq_pairs = build_equity(
    df_pair, pos_pairs,
    size_series=dynamic_size(df_pair, 0.008, 14, 1.5),
    alt_col='close_alt'
)
all_metrics['Pairs Trading'] = compute_metrics(eq_pairs, pos_pairs)
plot_strategy(df_pair, pos_pairs, eq_pairs,
              all_metrics['Pairs Trading'], 'pairs_trading', ['BTC', 'ETH'])

---
## Strategy 4 — Dual Momentum

### Concept
Combines two momentum timeframes: a **macro trend filter** (SMA50 vs SMA200 — the classic 'golden/death cross') with **micro momentum confirmation** (MACD histogram and EMA9 price position). Position size scales with ADX — stronger trends get larger allocations.

### Buy Signal (ALL conditions must be true)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Golden Cross | SMA50 > SMA200 | Macro bull regime |
| RSI | 35 < RSI < 72 | Momentum present, not overbought |
| MACD Histogram | hist > 0 | Short-term momentum positive |
| EMA9 | Close > EMA9 | Micro-trend bullish |

### Sell Signal (ANY condition triggers exit)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Death Cross | SMA50 < SMA200 | Macro bear regime |
| RSI | RSI > 82 | Overbought exhaustion |
| MACD hist | hist < hist_mean − 1σ | Short-term momentum collapse |

### Sizing
ADX-scaled: `size = base_size × (ADX / 30)`, clipped to [0.5×, 2.0×]. Strong trends get bigger positions.

In [ ]:
print('[4/9] DUAL MOMENTUM')
df = btc.copy()
df['sma50']  = df.close.rolling(50).mean()
df['sma200'] = df.close.rolling(200).mean()
df['ema9']   = df.close.ewm(span=9).mean()
df['rsi']    = compute_rsi(df.close, 14)
_, _, mh     = compute_macd(df.close)
df['mh']     = mh
adx, _, _    = compute_adx(df, 14)
df['adx']    = adx.fillna(20)   # fillna prevents NaN propagation into sizes

# ── BUY: macro + micro momentum aligned ─────────────────────────────
bc = ((df.sma50 > df.sma200)
      & df.rsi.between(35, 72)
      & (df.mh > 0)
      & (df.close > df.ema9))

# ── SELL: macro regime shift or momentum collapse ───────────────────
sc = ((df.sma50 < df.sma200)
      | (df.rsi > 82)
      | (df.mh < df.mh.rolling(20).mean() - df.mh.rolling(20).std()))

pos = build_positions(bc.astype(int).values, sc.astype(int).values)
sz  = dynamic_size(df, 0.010, 14, 2.0)
adx_scale = (df.adx / 30).clip(0.5, 2.0).fillna(1.0)
sz = (sz * adx_scale.values).clip(upper=3)   # ADX-scaled position sizing

eq = build_equity(df, pos, size_series=sz)
all_metrics['Dual Momentum'] = compute_metrics(eq, pos)
plot_strategy(df, pos, eq, all_metrics['Dual Momentum'], 'dual_momentum', ['BTC/USDT'])

---
## Strategy 5 — Margin Short

### Concept
A counter-trend strategy that shorts BTC only during confirmed **bear market regimes** (price below SMA200). In bull markets (2023–2024) this strategy stays flat — only 2 trades — which is the correct behaviour. It's designed to profit from bear-market bounces that fade.

> **Note**: Low trade count is intentional. With only 2 trades in a 2023+ dataset, the strategy requires a longer historical period (2021–2022 bear market) to show its edge.

### Short Entry (ALL conditions must be true)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Bear regime | Close < SMA200 | Only short in bear markets |
| RSI | RSI > 72 | Overbought bounce in downtrend |
| BB upper | Close > BB_upper × 0.98 | Price at upper band |
| MACD | hist declining | Momentum fading |
| 2 down closes | ≥2 of last 3 bars lower | Fakeout filter |

### Cover Signal (ANY condition triggers cover)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| RSI | RSI < 55 | Oversold — take profit |
| BB lower | Close < BB_lower × 1.02 | Reached lower band target |
| Regime flip | Close > SMA200 | Bear regime ended — cover |

### Sizing
0.9% risk, 2.2× ATR — slightly tighter than long strategies given short-side risk.

In [ ]:
print('[5/9] MARGIN SHORT')
df = btc.copy()
df['rsi']    = compute_rsi(df.close, 14)
df['sma200'] = df.close.rolling(200).mean()
bb_up, _, bb_lo = compute_bbands(df.close, 20, 2)
_, _, mh     = compute_macd(df.close)
df['mh']     = mh.fillna(0)
df['cdn']    = (df.close < df.close.shift(1)).astype(int).rolling(3).sum()
bear         = df.close < df.sma200   # bear regime filter

# ── SHORT: overbought bounce in bear regime ──────────────────────────
short_e = (bear
           & (df.rsi > 72)
           & (df.close > bb_up * 0.98)
           & (df.mh < df.mh.shift(1))
           & (df.cdn >= 2))

# ── COVER: oversold / BB lower / regime flip ────────────────────────
cover = (df.rsi < 55) | (df.close < bb_lo * 1.02) | (~bear)

pos = -build_positions(short_e.astype(int).values, cover.astype(int).values)
eq  = build_equity(df, pos, size_series=dynamic_size(df, 0.009, 14, 2.2))
all_metrics['Margin Short'] = compute_metrics(eq, pos)
plot_strategy(df, pos, eq, all_metrics['Margin Short'], 'margin_short', ['BTC/USDT'])

---
## Strategy 6 — Volatility Straddle (BB Squeeze Breakout)

### Concept
When volatility contracts (Bollinger Band width narrows), it often precedes a large directional move. This strategy enters long when volatility **expands** above a threshold, Bollinger Bands are widening, and price is above the midband.

It runs on **ETH** which historically shows sharper volatility breakouts than BTC.

### Buy Signal (ALL conditions must be true)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Vol spike | ATR / ATR_MA(200) > 1.8 | Volatility significantly elevated |
| BB widening | BB_width > BB_width(3 bars ago) | Bands actively expanding |
| Direction | Close > BB_midband | Breakout to the upside |

### Sell Signal (ANY condition triggers exit)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Vol normalises | vol_spike < 1.2 | Breakout move over |
| Bands contracting | BB_width < BB_width(6 bars ago) | Squeeze resuming |

### Sizing
1.1% risk, 1.8× ATR — moderate size since we're entering after vol expansion has begun.

In [ ]:
print('[6/9] VOL STRADDLE')
df = eth.copy()
df['atr']  = compute_atr(df, 14)
df['vs']   = (df.atr / df.atr.rolling(200).mean()).fillna(1.0)
bu, bm, bl = compute_bbands(df.close, 20, 2)
df['bw']   = ((bu - bl) / bm).fillna(0)

# ── BUY: vol expansion + BB widening + upside breakout ──────────────
bc = (df.vs > 1.8) & (df.bw > df.bw.shift(3)) & (df.close > bm)

# ── SELL: vol normalising or bands contracting ──────────────────────
sc = (df.vs < 1.2) | (df.bw < df.bw.shift(6))

pos = build_positions(bc.astype(int).values, sc.astype(int).values)
eq  = build_equity(df, pos, size_series=dynamic_size(df, 0.011, 14, 1.8))
all_metrics['Vol Straddle'] = compute_metrics(eq, pos)
plot_strategy(df, pos, eq, all_metrics['Vol Straddle'], 'vol_straddle', ['ETH/USDT'])

---
## Strategy 7 — Perpetual Swap Hedge (Delta-Neutral Funding Harvest)

### Concept
In bull markets, perpetual futures funding rates are persistently positive — long positions pay shorts every 8 hours. A **delta-neutral** position (long spot + short perp) earns this funding as pure yield with near-zero market exposure.

We model the hourly funding rate using RSI as a market-heat proxy:
```
ann_funding = 0.50 × max((RSI/50)² − 1, 0)
fr_hourly   = ann_funding / 8760
```
- RSI = 70 → ~48%/yr annualised → ~0.0055%/hr
- RSI = 60 → ~22%/yr annualised → ~0.0025%/hr
- RSI = 50 → 0%/yr (neutral market)

P&L = `position_lag × fr_hourly − 5bps × |trade|`

### Enter Hedge (ALL conditions)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Funding rate | fr_ma > 15%/yr | Worth harvesting after costs |
| Bull regime | Close > EMA50 > SMA200 | Only positive funding in uptrend |

### Close Hedge (ANY condition)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Funding dries up | fr_ma < 5%/yr | No longer worth the friction |
| Trend breaks | Close < EMA50 × 0.97 | Regime change risk |

### Risk Profile
- **Market risk**: Near-zero (delta-neutral)
- **Basis risk**: Perp may temporarily diverge from spot
- **Liquidation risk**: If spot gaps down, short perp collateral at risk

In [ ]:
print('[7/9] PERPETUAL SWAP HEDGE')
df = btc.copy()
df['rsi']    = compute_rsi(df.close, 14).fillna(50)
df['ema20']  = df.close.ewm(span=20).mean()
df['ema50']  = df.close.ewm(span=50).mean()
df['sma200'] = df.close.rolling(200).mean()

# Annualised funding proxy via RSI market-heat model
# RSI=70 → 48%/yr  |  RSI=60 → 22%/yr  |  RSI=50 → 0%/yr
df['ann_fr'] = 0.50 * ((df.rsi / 50) ** 2 - 1.0).clip(lower=0)
df['fr_hr']  = df.ann_fr / 8760                        # per-bar (1h) rate
df['fr_ma']  = df.fr_hr.rolling(72).mean().fillna(0)   # 3-day smoothed

# Confirmed uptrend: price > EMA50 > SMA200
uptrend = (df.close > df.ema50) & (df.ema50 > df.sma200.fillna(df.ema50))

# ── ENTER HEDGE: positive funding + bull regime ──────────────────────
enter = (df.fr_ma > (0.15 / 8760)) & uptrend

# ── EXIT HEDGE: funding dried up or regime broke ────────────────────
exit_ = (df.fr_ma < (0.05 / 8760)) | (df.close < df.ema50 * 0.97)

pos = build_positions(enter.astype(int).values, exit_.astype(int).values)

# Delta-neutral P&L: earn funding rate only, no market exposure
pos_lag = np.roll(pos, 1); pos_lag[0] = 0
pl      = pos_lag * df.fr_hr.values
pl     -= np.abs(np.diff(pos, prepend=0)) * 0.0005     # 5 bps per round-trip
eq      = pd.Series(np.cumprod(1 + pl), index=df.index)

all_metrics['Perp Swap Hedge'] = compute_metrics(eq, pos)
plot_strategy(df, pos, eq, all_metrics['Perp Swap Hedge'],
              'perp_swap_hedge', ['BTC Perp (Delta-Neutral)'])

---
## Strategy 8 — Inverse Perpetual Hedge

### Concept
BTC-margined ("inverse") perpetuals have a natural convexity property: when BTC falls, the USD value of the collateral also falls, creating a **second-order loss** on a short position — but *gaining* on the short notional. Holding `1.0× long spot + 0.5× short inverse perp` creates a hedge that:
- Reduces delta from 1.0 to 0.5 when hedged
- Benefits from convexity on large down-moves
- Earns positive funding when spot is overbought

### Hedge Entry (ALL conditions)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Overbought | RSI > 70 | Elevated correction risk |
| BB upper | Close > BB(2.5σ) × 0.995 | Price at extreme upper band |
| Bull regime | Close > SMA200 | Only hedge in uptrend (not exit) |

### Close Hedge (ANY condition)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| RSI cools | RSI < 50 | Correction risk passed |
| Trend support | Close < EMA50 | Broader pullback — remove hedge |

### P&L Formula
```
P&L = spot_return + hedge_pos × HEDGE_RATIO × (−spot_return) − costs
    = spot_return × (1 − 0.5 × hedge_pos)
```
When hedged: captures 50% of upside, limits 50% of downside.

In [ ]:
print('[8/9] INVERSE PERP HEDGE')
df = btc.copy()
df['rsi']    = compute_rsi(df.close, 14)
df['ema50']  = df.close.ewm(span=50).mean()
df['sma200'] = df.close.rolling(200).mean()
bu, _, _     = compute_bbands(df.close, 20, 2.5)
spot_r       = df.close.pct_change().fillna(0)

# ── HEDGE ENTRY: overbought + upper band + bull regime ──────────────
se = (df.rsi > 70) & (df.close > bu * 0.995) & (df.close > df.sma200)

# ── HEDGE EXIT: RSI cooled or trend support level reached ───────────
sx = (df.rsi < 50) | (df.close < df.ema50)

shp = -build_positions(se.astype(int).values, sx.astype(int).values)

HEDGE = 0.5   # hedge 50% of spot exposure
pl    = spot_r.values + np.roll(shp, 1) * HEDGE * (-spot_r.values)
pl[0] = 0
pl   -= np.abs(np.diff(shp, prepend=0)) * 0.0005
eq    = pd.Series(np.cumprod(1 + pl), index=df.index)

all_metrics['Inverse Perp Hedge'] = compute_metrics(eq, shp)
plot_strategy(df, shp, eq, all_metrics['Inverse Perp Hedge'],
              'inverse_perp_hedge', ['BTC Inverse Hedge (50%)'])

---
## Strategy 9 — Trend + Volatility Filter

### Concept
A clean, binary IN/OUT trend strategy that **steps aside during volatility explosions** rather than trying to hedge through them. This directly addresses the failure mode of the previous Synthetic Put strategy, which amplified losses on every down bar via a `1.5×` multiplier — catastrophic in live markets with consecutive down moves.

> **Why Synthetic Put failed live**: The old formula was `P&L = spot_ret × (1−δ) + δ × down × 1.5`. The `1.5×` amplifier fires on **every single down bar**. In a 3-bar decline of −1% each, the strategy loses ~4.5% instead of ~3%. In a real bear leg this compounds to -4000%+ drawdown.

### Design Principle
Vol spikes are the enemy of leveraged/amplified strategies. Instead of hedging vol, this strategy **avoids it entirely** — exiting before or as vol explodes, then re-entering once conditions normalise.

### Buy Signal (ALL conditions must be true)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Full EMA stack | EMA20 > EMA50 > EMA200 | All timeframes aligned bullish |
| Vol calm | ATR / ATR_MA(200) < 2.0 | Not entering a vol explosion |
| RSI | 40 < RSI < 70 | Trend momentum, not exhausted |
| MACD hist | hist > 0 | Short-term momentum positive |

### Sell Signal (ANY condition triggers exit)
| Condition | Threshold | Rationale |
|-----------|-----------|----------|
| Trend break | EMA20 < EMA50 | Short-term trend fails first |
| Vol explosion | ATR / ATR_MA > 2.5 | Step aside — high vol kills returns |
| RSI | RSI > 80 | Overbought exhaustion |
| MACD reversal | hist < −1σ(50-bar) | Momentum reversal confirmed |

### Why This Works Live
- **Binary position**: always 0 or 1, never amplified exposure
- **Vol exit**: the `vs > 2.5` condition exits *before* the big down moves materialise
- **EMA20 tripwire**: catches trend breaks early without being noisy
- **No per-bar formulas**: P&L = standard `position_lag × price_return` — identical in backtest and live

In [ ]:
print('[9/9] TREND + VOL FILTER')
df = btc.copy()
df['ema20']  = df.close.ewm(span=20).mean()
df['ema50']  = df.close.ewm(span=50).mean()
df['ema200'] = df.close.ewm(span=200).mean()
df['atr']    = compute_atr(df, 14)
df['atr_ma'] = df.atr.rolling(200).mean()
df['vs']     = (df.atr / df.atr_ma).fillna(1.0)  # vol spike ratio
df['rsi']    = compute_rsi(df.close, 14)
_, _, mh     = compute_macd(df.close)
df['mh']     = mh.fillna(0)
mh_thresh    = df.mh.rolling(50).std().fillna(0.001)  # adaptive MACD threshold

# ── BUY: full trend stack + calm vol + positive momentum ────────────
bc = ((df.ema20 > df.ema50)
      & (df.ema50 > df.ema200)
      & (df.vs < 2.0)
      & df.rsi.between(40, 70)
      & (df.mh > 0))

# ── SELL: trend break OR vol explosion OR exhaustion ────────────────
sc = ((df.ema20 < df.ema50)         # short-term trend breaks first
      | (df.vs > 2.5)               # vol explosion — step aside immediately
      | (df.rsi > 80)               # overbought exhaustion
      | (df.mh < -mh_thresh))       # MACD reversal confirmed

pos = build_positions(bc.astype(int).values, sc.astype(int).values)
eq  = build_equity(df, pos, size_series=dynamic_size(df, 0.010, 14, 2.0))
all_metrics['Trend+Vol Filter'] = compute_metrics(eq, pos)
plot_strategy(df, pos, eq, all_metrics['Trend+Vol Filter'],
              'trend_vol_filter', ['BTC/USDT'])

---
## 📊 Performance Summary

The table below colour-codes each metric:
- **ROI / CAGR**: Green = positive, Red = negative
- **Sharpe**: Green > 1.0, Yellow > 0, Red < 0
- **Max DD**: Green < 10%, Yellow < 25%, Red ≥ 25%
- **Win Rate**: Green > 55%, Yellow > 45%, Red < 45%

In [ ]:
apply_style()
sdf  = pd.DataFrame(all_metrics).T
cols = ['ROI (%)', 'CAGR (%)', 'Sharpe', 'Max DD (%)', 'Win Rate (%)', 'Num Trades']
sdf  = sdf[cols]
for col in cols:
    sdf[col] = pd.to_numeric(sdf[col], errors='coerce')

fig, ax = plt.subplots(figsize=(15, 0.8 + len(sdf) * 0.65), facecolor=S['bg'])
ax.set_facecolor(S['bg']); ax.axis('off')
tbl = ax.table(
    cellText=sdf.round(2).values,
    rowLabels=sdf.index,
    colLabels=[c.replace(' ', '\n') for c in cols],
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.1, 2.1)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor(S['border'])
    if r == 0:
        cell.set_facecolor('#1e2d4a'); cell.set_text_props(color=S['bright'], weight='bold')
    elif c == -1:
        cell.set_facecolor('#0a0a0a'); cell.set_text_props(color=S['blue'], weight='bold')
    else:
        try:
            v = float(sdf.round(2).values[r-1][c]); cn = cols[c]
        except: continue
        if np.isnan(v): fc = S['panel']; tc = S['text']
        elif cn in ['ROI (%)', 'CAGR (%)']:
            fc = '#14532d' if v > 0 else '#450a0a'; tc = S['green'] if v > 0 else S['red']
        elif cn == 'Sharpe':
            fc = '#14532d' if v > 1 else ('#78350f' if v > 0 else '#450a0a')
            tc = S['green'] if v > 1 else (S['yellow'] if v > 0 else S['red'])
        elif cn == 'Max DD (%)':
            fc = '#450a0a' if v < -25 else ('#78350f' if v < -10 else '#14532d')
            tc = S['red'] if v < -25 else (S['yellow'] if v < -10 else S['green'])
        elif cn == 'Win Rate (%)':
            fc = '#14532d' if v > 55 else ('#78350f' if v > 45 else '#450a0a')
            tc = S['green'] if v > 55 else (S['yellow'] if v > 45 else S['red'])
        else: fc = S['panel']; tc = S['text']
        cell.set_facecolor(fc); cell.set_text_props(color=tc, weight='bold')

ax.set_title('◆  STRATEGY PERFORMANCE SUMMARY  ◆', color=S['bright'],
             fontsize=15, weight='bold', pad=18)
plt.tight_layout()
plt.savefig('ui/public/backtest_results/summary_table.png',
            format='png', dpi=150, bbox_inches='tight', facecolor=S['bg'])
plt.close(fig)

print('\n' + '═'*70)
print('  PERFORMANCE SUMMARY')
print('═'*70)
print(sdf.to_string(float_format=lambda x: f'{x:8.2f}'))
print('═'*70)
print('\n✅ Done — charts saved to ui/public/backtest_results/')